# A/B Test - Power Analysis

**Objective:** Calculate required sample size and analyze test power.

**Author:** Ashish Patel  
**Date:** 2026-05-26

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries loaded')

## 1. Define Parameters

In [ ]:
# Test parameters
baseline_rate = 0.125  # Current conversion rate
mde = 0.02  # Minimum detectable effect (absolute)
alpha = 0.05  # Significance level
power = 0.80  # Desired power

print('📊 Test Parameters')
print('=' * 50)
print(f'Baseline conversion rate: {baseline_rate*100:.1f}%')
print(f'Minimum detectable effect: {mde*100:.1f}%')
print(f'Expected treatment rate: {(baseline_rate + mde)*100:.1f}%')
print(f'Significance level (α): {alpha}')
print(f'Desired power (1-β): {power}')

## 2. Sample Size Calculation

In [ ]:
# Calculate effect size
effect_size = proportion_effectsize(baseline_rate, baseline_rate + mde)
print(f'Effect size (Cohen\'s h): {effect_size:.4f}')

# Calculate sample size
power_analysis = NormalIndPower()
sample_size = power_analysis.solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=power,
    ratio=1.0,
    alternative='two-sided'
)

sample_size_per_group = int(np.ceil(sample_size))
total_sample = sample_size_per_group * 2

print(f'\n📊 Required Sample Size')
print('=' * 50)
print(f'Per group: {sample_size_per_group:,}')
print(f'Total: {total_sample:,}')

## 3. Test Duration Estimate

In [ ]:
# Estimate duration based on traffic
daily_visitors = 5000  # Example daily traffic
traffic_allocation = 1.0  # 100% in test

users_per_day = daily_visitors * traffic_allocation
days_needed = int(np.ceil(total_sample / users_per_day))

print('📅 Duration Estimate')
print('=' * 50)
print(f'Daily visitors: {daily_visitors:,}')
print(f'Traffic in test: {traffic_allocation*100:.0f}%')
print(f'Users per day: {users_per_day:,.0f}')
print(f'Days needed: {days_needed}')
print(f'Weeks needed: {days_needed/7:.1f}')

## 4. Power Curve

In [ ]:
# Power vs sample size
sample_sizes = np.arange(100, 10000, 100)
powers = []

for n in sample_sizes:
    p = power_analysis.solve_power(
        effect_size=effect_size,
        nobs1=n,
        alpha=alpha,
        ratio=1.0,
        alternative='two-sided'
    )
    powers.append(p)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(sample_sizes, powers, linewidth=2)
ax.axhline(y=0.8, color='red', linestyle='--', label='80% power')
ax.axvline(x=sample_size_per_group, color='green', linestyle='--', label=f'n={sample_size_per_group:,}')
ax.set_xlabel('Sample Size (per group)')
ax.set_ylabel('Power')
ax.set_title('Power Curve', fontweight='bold')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../docs/img/power_analysis.png', dpi=150)
plt.show()

## 5. MDE vs Sample Size

In [ ]:
# What effect can we detect with different sample sizes?
sample_sizes = [1000, 2500, 5000, 10000, 25000, 50000]
mdes = []

for n in sample_sizes:
    es = power_analysis.solve_power(
        nobs1=n,
        alpha=alpha,
        power=power,
        ratio=1.0,
        alternative='two-sided'
    )
    # Convert back to absolute difference
    mde_abs = 2 * np.arcsin(np.sqrt(baseline_rate + es/2)) - 2 * np.arcsin(np.sqrt(baseline_rate))
    mdes.append(es)

print('📊 Minimum Detectable Effect by Sample Size')
print('=' * 50)
for n, es in zip(sample_sizes, mdes):
    print(f'n={n:,} per group → MDE = {es:.4f} effect size')

## 6. Summary

In [ ]:
print('=' * 60)
print('📊 POWER ANALYSIS SUMMARY')
print('=' * 60)
print(f"""
TEST DESIGN:
• Baseline rate: {baseline_rate*100:.1f}%
• MDE: {mde*100:.1f}% (absolute)
• Significance: α = {alpha}
• Power: {power*100:.0f}%

REQUIREMENTS:
• Sample per group: {sample_size_per_group:,}
• Total sample: {total_sample:,}
• Estimated duration: {days_needed} days

RECOMMENDATIONS:
• Run test for at least 1 full week to capture weekly patterns
• Don't peek at results early (inflates false positive rate)
• Consider sequential testing if duration is critical
""")